# Morpheme Tokenizer — Hybrid Neural + FST Morpheme Segmentation

Turkic languages are agglutinative: a single word can carry many suffixes
encoding grammatical features like number, case, possession, tense, and more.
Understanding the internal morpheme structure of words is critical for:

- **Morphology-aware NLP:** better tokenisation for language models
- **Linguistic analysis:** automatic morpheme glossing
- **Educational tools:** breaking words into meaningful units
- **Low-resource MT:** morpheme-level translation strategies

TurkicNLP's `MorphemeTokenizer` uses a **hybrid approach**:

1. **Neural backbone** (Glot500 morph model) — provides UPOS, UD features, and lemma
2. **Apertium HFST transducer** — adds derivational morphology tags
3. **Language-specific suffix tables** — maps UD features to surface allomorphs
   using phonological rules (vowel harmony, consonant context)

**Supported languages (16):** Turkish, Azerbaijani, Kazakh, Uzbek, Kyrgyz,
Tatar, Bashkir, Turkmen, Crimean Tatar, Sakha, Khakas, Tuvan, Southern Altai,
Northern Altai, Chuvash, Gagauz

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp.processors.morpheme_tokenizer import MorphemeTokenizer

## 1. Basic Usage — Kazakh

In [ ]:
tok = MorphemeTokenizer(lang="kaz")
tok.load()

# "houses" — stem + plural suffix
result = tok.segment("үйлер")
print(f"Word:     {result.word}")
print(f"Segments: {result.segments}")
print(f"Labels:   {[m.label for m in result.morphemes]}")
print(f"Labeled:  {result.labeled}")

In [ ]:
# More complex Kazakh examples
words = [
    ("мектепке", "to school — stem + dative"),
    ("баладан", "from child — stem + ablative"),
    ("кітабым", "my book — stem + possessive"),
    ("үйлеріңізде", "in your (formal) houses — stem + plural + poss + locative"),
    ("бардым", "I went — stem + past tense + 1sg"),
]

print(f"{'Word':<25} {'Segments':<35} {'Labels'}")
print("-" * 80)
for word, gloss in words:
    result = tok.segment(word)
    segs = " + ".join(result.segments)
    labs = " + ".join(m.label for m in result.morphemes)
    print(f"{word:<25} {segs:<35} {labs}")
    print(f"  {'':25} ({gloss})")

## 2. Turkish Morpheme Segmentation

In [ ]:
tok_tur = MorphemeTokenizer(lang="tur")
tok_tur.load()

words = [
    ("evlerde", "in houses — stem + plural + locative"),
    ("gidiyordum", "I was going — stem + progressive + past + 1sg"),
    ("okumuşlardır", "they have read — stem + evidential + plural + copula"),
    ("güzelleştirilmek", "to be beautified — stem + become + causative + passive + infinitive"),
    ("kitaplarımızdan", "from our books — stem + plural + possessive + ablative"),
]

print(f"{'Word':<25} {'Segments':<40} {'Labels'}")
print("-" * 90)
for word, gloss in words:
    result = tok_tur.segment(word)
    segs = " + ".join(result.segments)
    labs = " + ".join(m.label for m in result.morphemes)
    print(f"{word:<25} {segs:<40} {labs}")
    print(f"  {'':25} ({gloss})")

## 3. Comparing Morpheme Segmentation Across Languages

The same grammatical concept (e.g., plural + dative) is expressed with different allomorphs across Turkic languages due to vowel harmony and consonant assimilation rules.

In [ ]:
# "to schools" in different Turkic languages
examples = [
    ("tur", "okullara",    "Turkish"),
    ("kaz", "мектептерге", "Kazakh"),
    ("uzb", "maktablarga", "Uzbek"),
    ("kir", "мектептерге", "Kyrgyz"),
    ("tat", "мәктәпләргә", "Tatar"),
    ("aze", "məktəblərə",  "Azerbaijani"),
]

for lang, word, label in examples:
    tok = MorphemeTokenizer(lang=lang)
    tok.load()
    result = tok.segment(word)
    segs = " + ".join(result.segments)
    labs = " + ".join(m.label for m in result.morphemes)
    print(f"[{label:<12}] {word:<20} → {segs}")
    print(f"{'':16} labels: {labs}")

## 4. Processing Full Sentences

The `MorphemeTokenizer` can also process entire documents via its `.process()` method, which adds morpheme annotations to each word.

In [ ]:
from turkicnlp import Pipeline

# First create a pipeline to tokenize the text
nlp = Pipeline("kaz", processors=["tokenize", "morph_neural"])
doc = nlp("Мен мектепке бардым.")

# Then apply morpheme tokenizer to each word
tok_kaz = MorphemeTokenizer(lang="kaz")
tok_kaz.load()
doc = tok_kaz.process(doc)

print(f"{'Word':<20} {'Morphemes':<35} {'Labels'}")
print("-" * 70)
for w in doc.words:
    # _morphemes is set by MorphemeTokenizer.process() on each Word
    morphemes = getattr(w, '_morphemes', None)
    if morphemes:
        segs = " + ".join(m.text for m in morphemes)
        labs = " + ".join(m.label for m in morphemes)
    else:
        segs = w.text
        labs = "STEM"
    print(f"{w.text:<20} {segs:<35} {labs}")

## 5. Vowel Harmony in Action

One of the key features of the morpheme tokenizer is its awareness of phonological rules. The same suffix has different surface forms depending on the vowel harmony class of the stem.

In [ ]:
# Turkish dative suffix: -a/-e (palatal harmony)
# Turkish plural suffix: -lar/-ler (palatal harmony)
tok_tur2 = MorphemeTokenizer(lang="tur")
tok_tur2.load()

harmony_examples = [
    ("evlere",     "to houses — front vowel stem → -ler, -e"),
    ("okullara",   "to schools — back vowel stem → -lar, -a"),
    ("kitaplarda",  "in books — back vowel stem → -lar, -da"),
    ("defterlerde", "in notebooks — front vowel stem → -ler, -de"),
]

for word, note in harmony_examples:
    result = tok_tur2.segment(word)
    segs = " + ".join(result.segments)
    print(f"{word:<20} → {segs}")
    print(f"{'':20}   ({note})")